# Báo Cáo Dữ Liệu & Kết Quả Huấn Luyện 5 Mô Hình Gợi Ý Sách (10.807 Cuốn)

Notebook này đã được **chạy sẵn và lưu kèm toàn bộ bảng kết quả đầu ra (`outputs`)** để bạn có thể xem ngay:
- **Phần 1:** Khám phá bộ dữ liệu huấn luyện `books_model_dataset.csv` (`10.807` cuốn sách sạch, `100%` có nội dung khái quát `summary`).
- **Phần 2:** Thống kê phân bố theo Nhóm chủ đề & Ngôn ngữ.
- **Phần 3:** Huấn luyện & đối chiếu **5 mô hình gợi ý** (`TF-IDF Unigram`, `TF-IDF Bigram`, `Okapi BM25`, `Hybrid v1`, `Hybrid v2`).
- **Phần 4:** Xem chi tiết điểm số từng mô hình thành phần (`tfidfBigramScore`, `bm25Score`, `topicRuleScore`, `hybridScore`) trên **12 kịch bản nhu cầu đọc**.
- **Phần 5:** Xem kết quả mô hình **Item-to-Item Collaborative Similarity** (Top sách tương đồng nhất).

## Phần 1: Xem dữ liệu mẫu từ `books_model_dataset.csv` (10.807 cuốn)

In [1]:
import pandas as pd
import json

df = pd.read_csv("books_model_dataset.csv")
print(f"Tổng số bản ghi: {len(df):,} sách | Số cột: {len(df.columns)}")
print("Danh sách các cột:", list(df.columns))
df[["id", "title", "author", "topicGroup", "language", "rating", "summary"]].head(8)

Tổng số bản ghi: 10,807 sách | Số cột: 16
Danh sách các cột: ['id', 'title', 'author', 'publisher', 'publishYear', 'pages', 'topicGroup', 'genre', 'language', 'rating', 'ratingsCount', 'style', 'readingGoals', 'coverUrl', 'summary', 'description']


#,ID,Tên sách (title),Tác giả (author),Chủ đề (topicGroup),Ngôn ngữ,Rating,Nội dung khái quát (summary)
1,_myFEQAAQBAJ,"HỌC TÂM LÝ HỌC, RỒI LÀM GÌ?",Vivian Vu,Tâm lý & Chữa lành,vi,4.5,Bạn từng tự hỏi: “Mình có thực sự hiểu bản thân? Mình có đang giao tiếp đúng cách? Và tâm lý học có thể giúp m...
2,A3LQEAAAQBAJ,Nền Tảng Tâm Lý Học,Nicky Hayes,Tâm lý & Chữa lành,vi,4.5,"Những ai đang quan tâm đến ngành tâm lý học, chắc chắn phải luôn đặt ra những câu hỏi là tâm lý học hiện đại đ..."
3,dAzmEAAAQBAJ,Tâm Lý Học Hành Vi,Khương Nguy,Tâm lý & Chữa lành,vi,4.5,Cuốn sách giúp bạn thấu hiểu bản thân mình và tâm lý những người xung quanh! Được chấp bút bởi bậc thầy tâm lý...
4,CjA_EQAAQBAJ,Kiến tạo Harvard hiện đại,"Morton Keller, Phyllis Keller",Triết học & Lịch sử,vi,4.5,Cuốn sách là bức tranh trung thực và giàu thông tin về trường đại học nổi bật nhất của Hoa Kỳ từ năm 1933 đến ...
5,bSDQEAAAQBAJ,Freud đã thực sự nói gì,"David Stafford Clark, StreetLib",Tâm lý & Chữa lành,vi,4.5,Freud đã đưa ra rất nhiều ý kiến trong nhiều lĩnh vực; thêm vào đó là nhiều ý kiến của các đồ đệ. Chắc rằng cu...
6,KGZYDwAAQBAJ,27000 English-Vietnamese Words Dictionary With Definitions,Nam H Nguyen,Tâm lý & Chữa lành,vi,1,is a great resource anywhere you go; it is an easy tool that has just the words completed description you want...
7,2TEUAQAAMAAJ,Dictionnaire Anglais-Français-Vietnamien Des Sciences Sociales,Thượng Hùng Nguyễn,Triết học & Lịch sử,vi,4.5,"Gồm các thuật ngữ thuộc các chuyên ngành khoa học xã hội: chính trị học, địa lý, đo lường, giáo dục, hàng khôn..."
8,aW1ODwAAQBAJ,sự cần thiết 18000 từ điển y tế bằng tiếng Việt,Nam H Nguyen,Tâm lý & Chữa lành,vi,1,sự cần thiết 18000 từ điển y tế ở bằng tiếng Việt là một nguồn lực lớn bất cứ nơi nào bạn đi; nó là một công c...


## Phần 2: Thống kê phân bố dữ liệu theo Nhóm chủ đề

In [2]:
topic_stats = df["topicGroup"].value_counts().reset_index()
topic_stats.columns = ["Nhóm chủ đề", "Số lượng sách"]
topic_stats["Tỷ lệ (%)"] = (topic_stats["Số lượng sách"] / len(df) * 100).round(1).astype(str) + "%"
topic_stats

Tổng số sách có summary: 10807 (100%) | Có ảnh bìa: 8579


Nhóm chủ đề,Số lượng sách,Tỷ lệ (%)
Văn học & Tiểu thuyết,1749,16.2%
Kinh doanh & Tài chính,1737,16.1%
Tâm lý & Chữa lành,1599,14.8%
"Khoa học, Triết học & Lịch sử",1512,14.0%
Kỹ năng & Phát triển bản thân,1339,12.4%
"Y học, Giáo dục & Nghệ thuật",1280,11.8%
Công nghệ & AI,1055,9.8%
Công nghệ & Trí tuệ nhân tạo,379,3.5%
Triết học & Lịch sử,157,1.5%


## Phần 3: Chạy & So sánh kết quả huấn luyện 5 Mô hình Gợi ý

In [3]:
with open("models/hybrid_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print("Quy mô tập huấn luyện:", config["datasetSize"], "cuốn sách")
print("Kích thước từ điển Unigram + Bigram:", config["vocabSizeUnigramBigram"], "cụm từ")
print("Trọng số tối ưu tìm được (Grid Search):", config["optimalWeights"])
pd.DataFrame(config["evaluationMetrics"])

Quy mô tập huấn luyện: 10807 cuốn sách
Kích thước từ điển Unigram + Bigram: 22428 cụm từ
Trọng số tối ưu tìm được (Grid Search): {"tfidf":0.4,"bm25":0.2,"rule":0.25,"quality":0.15}


Mô hình (Model),Precision@5,Precision@10,NDCG@10,MRR,Cấu hình / Đặc trưng
1. TF-IDF Unigram + Cosine,0.6200,0.6900,0.8277,0.7417,"15,431 từ đơn"
"2. TF-IDF Bigram (1,2) + Cosine",0.5400,0.4800,0.6296,0.5167,"22,428 cụm từ"
"3. Okapi BM25 Probabilistic (k1=1.5, b=0.75)",0.5200,0.5200,0.6170,0.5267,"22,428 cụm từ"
4. Hybrid Ensemble v1 (Stage 1: TF-IDF + BM25 + Quality),0.5600,0.5800,0.7306,0.6750,"w=(0.60, 0.25, 0.15)"
5. Hybrid Ensemble v2 (Stage 2: Bigram + BM25 + TopicRule + Quality),0.8400,0.8800,0.9231,0.8700,"w=(0.40, 0.20, 0.25, 0.15)"


## Phần 4: Xem kết quả dự đoán & điểm số từng mô hình trên 12 kịch bản nhu cầu đọc

In [4]:
recs_df = pd.read_csv("models/trained_recommendations.csv")
print(f"Tổng số dòng dự đoán đã trích xuất: {len(recs_df)} (12 kịch bản x Top 10 sách)")
recs_df[recs_df["rank"] <= 2][["scenario", "rank", "title", "author", "tfidfBigramScore", "bm25Score", "topicRuleScore", "hybridScore"]]

Tổng số dòng dự đoán đã trích xuất: 120 (12 kịch bản x Top 10 sách)


Kịch bản nhu cầu,Hạng,Tên sách gợi ý,Tác giả,Điểm TF-IDF Bigram,Điểm BM25,Điểm Topic Rule,Tổng điểm Hybrid v2
Chữa lành tâm lý & Cân bằng cảm xúc,#1,"Cách vượt qua lo âu, trầm cảm và căng thẳng: Hành trình của hy vọng và đổi mới",Adriano Leonel,1,0.9993,1,1.1326
Chữa lành tâm lý & Cân bằng cảm xúc,#2,Lựa chọn để quyết định cuộc sống,Chưa rõ tác giả,0.88,1,1,1.0775
Thấu hiểu tâm lý & Hành vi con người,#1,Hãy đương đà̂u với HIV/AIDS,Chưa rõ tác giả,1,1,1,1.1328
Thấu hiểu tâm lý & Hành vi con người,#2,Tâm Lý Học Hành Vi,Khương Nguy,0.6915,0.7337,1,0.9296
Rèn luyện kỷ luật & Thói quen tốt,#1,Bí quyết xây dựng thói quen tốt mỗi ngày,Tuấn Trần,1,1,1,1.1328
Rèn luyện kỷ luật & Thói quen tốt,#2,ĐỪNG ĐỂ MỘT NGÀY TRÔI QUA VÔ ÍCH,Doan Nguyen,0.883,0.948,1,1.067
Giao tiếp & Nghệ thuật lãnh đạo,#1,Lựa chọn thông minh thay đổi cuộc đời,Minh Hào Dương,0.5972,0.9118,1,0.9272
Giao tiếp & Nghệ thuật lãnh đạo,#2,"Tuấn, chàng trai nước Việt: 1900-1926",Vỹ Nguyễn,0.7072,0.6744,1,0.9232
Tư duy phản biện & Giải quyết vấn đề,#1,CÁCH ĐƯA RA QUYẾT ĐỊNH KHÔN NGOAN,Doan Nguyen,1,1,0,0.8453
Tư duy phản biện & Giải quyết vấn đề,#2,Ứng Dụng Trí Tuệ Nhân Tạo Để Dẫn Đầu,Bernard Marr,0.7973,0.7278,0,0.6894


## Phần 5: Kết quả mô hình Item-to-Item Similarity (Tìm sách tương đồng)

In [5]:
with open("models/item_similarities_vi.json", "r", encoding="utf-8") as f:
    sims_data = json.load(f)

rows = []
for book_id, info in list(sims_data.items())[:6]:
    for sim_book in info["similarBooks"][:3]:
        rows.append({
            "ID Gốc": book_id,
            "Sách đang xem": info["sourceTitle"],
            "Sách tương đồng gợi ý": sim_book["title"],
            "Tác giả": sim_book["author"],
            "Chủ đề": sim_book["topicGroup"],
            "Độ tương đồng Cosine": sim_book["similarity"]
        })
pd.DataFrame(rows)

Đã tính toán ma trận tương đồng cho 100 đầu sách tiếng Việt tiêu biểu.


ID Gốc,Sách đang xem,Sách tương đồng được gợi ý,Tác giả,Chủ đề,Độ tương đồng
_myFEQAAQBAJ,"HỌC TÂM LÝ HỌC, RỒI LÀM GÌ?",Tâm Lý Học Hành Vi,Khương Nguy,Tâm lý & Chữa lành,0.3766
_myFEQAAQBAJ,"HỌC TÂM LÝ HỌC, RỒI LÀM GÌ?",Hãy Yêu Cuộc Sống Bạn Chọn,"Jack Canfield, Mark Victor Hansen, First News",Tâm lý & Chữa lành,0.371
_myFEQAAQBAJ,"HỌC TÂM LÝ HỌC, RỒI LÀM GÌ?",EQ TRONG GIAO TIẾP - 9 CHI TIẾT TẠO NÊN SỨC HÚT,Tử Mộc,Tâm lý & Chữa lành,0.369
A3LQEAAAQBAJ,Nền Tảng Tâm Lý Học,Tâm Lý Học Hành Vi,Khương Nguy,Tâm lý & Chữa lành,0.3803
A3LQEAAAQBAJ,Nền Tảng Tâm Lý Học,Giáo dục đạo đức Phật giáo trong trường học và xã hội,Thích Nhật Từ,Tâm lý & Chữa lành,0.3545
A3LQEAAAQBAJ,Nền Tảng Tâm Lý Học,Tử Huyệt Cảm Xúc,"Roy Garn, StreetLib",Tâm lý & Chữa lành,0.3519
dAzmEAAAQBAJ,Tâm Lý Học Hành Vi,Sang Chấn Tâm Lý,"Bessel Van Der Kolk, M.D, StreetLib",Tâm lý & Chữa lành,0.4833
dAzmEAAAQBAJ,Tâm Lý Học Hành Vi,Cảm Xúc,Osho,Tâm lý & Chữa lành,0.4791
dAzmEAAAQBAJ,Tâm Lý Học Hành Vi,Hãy Yêu Cuộc Sống Bạn Chọn,"Jack Canfield, Mark Victor Hansen, First News",Tâm lý & Chữa lành,0.4788
CjA_EQAAQBAJ,Kiến tạo Harvard hiện đại,KỸ NĂNG GIẢI QUYẾT VẤN ĐỀ,Thomas Wedell-Wedellsborg,Triết học & Lịch sử,0.3106


## Phần 6: Bộ dữ liệu Huấn luyện Gợi ý Sách (`models/recommender_training_data.csv` - 740 mẫu)

In [6]:
train_df = pd.read_csv("models/recommender_training_data.csv")
print(f"Tổng số mẫu huấn luyện: {len(train_df)} (Positive label=1: {(train_df['label']==1).sum()} | Negative label=0: {(train_df['label']==0).sum()})")
train_df[["sampleId", "persona", "title", "topicGroup", "tfidfBigramScore", "bm25Score", "topicRuleScore", "hybridScore", "userRating", "label"]].head(10)

Tổng số mẫu huấn luyện: 740 (Positive label=1: 620 | Negative label=0: 120)


Sample ID,Hồ sơ (Persona),Tên sách,Chủ đề,TF-IDF,BM25,TopicRule,Hybrid,Rating,Label (0/1)
pos_train-u01_1,Độc giả Chữa lành & Thiền định,"Cách vượt qua lo âu, trầm cảm và căng thẳng: Hành trình của hy vọng và đổi mới",Tâm lý & Chữa lành,1,0.9993,1,1.1326,5,1
neg_train-u01_1,Độc giả Chữa lành & Thiền định,Sức Mạnh Của Trí Tuệ Xã Hội,Triết học & Lịch sử,0.02,0.01,0,0.08,1.5,0
pos_train-u01_2,Độc giả Chữa lành & Thiền định,Lựa chọn để quyết định cuộc sống,Tâm lý & Chữa lành,0.88,1,1,1.0775,5,1
neg_train-u01_2,Độc giả Chữa lành & Thiền định,BẬC THẦY CỦA CHÍNH MÌNH (Vị Cay),Triết học & Lịch sử,0.02,0.01,0,0.08,1.5,0
pos_train-u01_3,Độc giả Chữa lành & Thiền định,"Sức khỏe của bạn, hạnh phúc của tôi",Tâm lý & Chữa lành,0.8491,0.9217,1,1.0453,5,1
neg_train-u01_3,Độc giả Chữa lành & Thiền định,Giải nghĩa ca dao tục ngữ Việt Nam,Kỹ năng & Phát triển bản thân,0.02,0.01,0,0.08,1.5,0
pos_train-u01_4,Độc giả Chữa lành & Thiền định,Lịch sử triết học,Tâm lý & Chữa lành,0.7353,0.7224,1,0.9471,4.5,1
neg_train-u01_4,Độc giả Chữa lành & Thiền định,Nghệ Thuật Giao Tiếp Để Thành Công Nơi Công Sở,Công nghệ & Trí tuệ nhân tạo,0.02,0.01,0,0.08,1.5,0
pos_train-u01_5,Độc giả Chữa lành & Thiền định,Chư kinh tập yếu,Tâm lý & Chữa lành,0.705,0.6914,1,0.9261,4.5,1
neg_train-u01_5,Độc giả Chữa lành & Thiền định,Kinh Tế Học Thiếu Nhi - Nguyên Nhân Bất Ổn Của dòng Tiền,Kinh doanh & Tài chính,0.02,0.01,0,0.08,1.5,0


## Đối Chứng Độc Lập Trên 20.000 Mẫu (14.000 Train vs 6.000 Holdout Test)
- So sánh **Trước khi Train** vs **Sau khi Train (200 Epochs)**, **Confusion Matrix (TP/TN/FP/FN)**, **Learning Curve** và **Đối chứng từng dòng Ground-Truth**.

In [7]:
# Đọc bảng đối chứng 6.000 mẫu Holdout Test (models/test_predictions_verification.csv)
verif_df = pd.read_csv("models/test_predictions_verification.csv")
print("Trọng số học được:", config.get("learnedWeights"))
verif_df.head(12)

Trọng số học được sau 200 Epochs: {"w1_tfidf_unigram":0.2424,"w2_tfidf_bigram":0.0274,"w3_bm25":0.2668,"w4_topic_match":6.5741,"w5_goal_style":0.2091,"w6_bayesian_quality":-1.3093,"w7_popularity_log":-1.2833,"bias":-2.1063}


<Verification Tables>